# What is hidden inside a Martian meteorite?

You are operating the ODIN imaging instrument at ESS. Behind this page, a simulation of the instrument produces your
measurements, in the formats of the real instrument.

**What you will learn**

- why neutrons see the water in a rock, which X-rays hardly see;
- how a radiograph is made from three images (sample, open beam, dark);
- how a tomography gives the attenuation coefficient $\Sigma$ at every point inside;
- how $\Sigma$ tells what a hidden region is made of, and how much water it holds.


**Your mission.** A request for the Neutron Solutions Team of ESS, where you are a trainee, from the Mars team:
planetary scientists who study the **water of early Mars**. Mars once had rivers and lakes; part of that water was
taken into the rocks of the crust, bound in their minerals, and a rock keeps it for billions of years. The meteorite
NWA 7034 ("Black Beauty"), found in the Sahara in 2011, is a piece of that crust, 4.4 billion years old, and the
Martian meteorite richest in water: about 6 000 ppm.



*One horizontal slice of the piece (8 mm × 2 mm), seen from above. A radiograph adds up everything along the beam (left): two clasts one behind the other make one peak. Turning the sample and taking radiographs over 180° (middle) gives back each slice by filtered back-projection (right). The H-rich clasts, about 0.3 mm across, are drawn 3 times larger.*

The team's question: **where is this water held?** Spread evenly in the whole rock, or concentrated in some of the
rock fragments (clasts) that make it up? The two answers tell different stories: water spread everywhere points to
the whole rock having been soaked; water held in a few clasts means these fragments carried it with them, a reservoir
of their own, which counts in how much water the young crust of Mars could store. The team also wants to cut out a
tiny grain of the richest clasts for a chemical analysis of its hydrogen: they need to know where these clasts are.

A gram of the meteorite is worth more than a gram of gold: you must answer **without cutting it**. Find where the
water is, and how much the richest regions hold (part 6).

**The sample is real.** A piece of NWA 7034 of 12 × 8 × 2 mm³ was imaged with neutrons at the ICON instrument (PSI)
and with X-rays at the ESRF by E. B. Naver, K. W. Nikolajsen *et al.*, *Direct detection of hydrogen reveals a new
macroscopic crustal water reservoir on early Mars*, arXiv:2601.08390 (2026). The twin uses a model of that piece
built from their numbers: its size, its kinds of material, their sizes and amounts, and the neutron attenuations
they measured. You will redo their measurement on ODIN, and compare.

**Why neutrons.** X-rays are stopped by heavy atoms (iron, titanium) and hardly see hydrogen. Neutrons do the
opposite: **hydrogen** scatters them more than any other element, while silicon, oxygen, aluminum or iron let them
through. In a neutron image, the hydrogen of the water stands out.

**What you will measure.** A beam goes through the sample; the fraction that comes out, the **transmission**, is

$$T = \frac{I}{I_0} = e^{-\Sigma L}$$

with $L$ the thickness crossed and $\Sigma$ the **attenuation coefficient** of the material (in cm⁻¹). The camera
records $T$ for each pixel: a **radiograph** (parts 1 to 3). A radiograph adds up everything along the beam; turning
the sample and taking radiographs at many angles gives back $\Sigma$ at every point inside: a **tomography** (part 4).
The value of $\Sigma$ then tells what the material is (part 5).

**Questions along the page** check each step: answer in the box, then click *Check*. A wrong answer is explained;
after a try you can also show the answer. Your answers are kept.

In [ ]:
import sys, os, json, warnings
warnings.filterwarnings("ignore")
sys.path[:0] = [os.path.abspath("../../.."), os.path.abspath("../..")]     # twin/ and twin/odin/
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as w
from IPython.display import display, clear_output
from odin_virtual import VirtualOdin, Cancelled
import measure_control as mc          # Stop button, settings locked during a measurement, renaming a file
import analysis_chain                 # the folded "Under the hood" section
import odin_radiography as tomo
import odin_viz as viz
import live_view
from page_output import into
import web_user
web_user.sign_in()            # the online version: the anonymous student of the start page (nothing to do in the app;
                              # with pre-heated kernels the modules above are loaded before the student arrives)
import quiz
q = quiz.Quiz("odin_meteorite_bsc")      # the questions, along the page

SAMPLE = "meteorite_nwa7034"
odin = VirtualOdin(session=os.environ.get("ODIN_SESSION", "student"))
odin.move(sample=SAMPLE)
EXT = tomo.extent()
SETUPS = {"standard": (128, 0.03), "high resolution": (512, 0.01)}     # camera pixels, pinhole

## 0. Your sample

A sawn piece **12 mm high, 8 mm wide and 2 mm thick**, its edges slightly irregular and one corner broken, glued on
an aluminium pin on the rotation stage of ODIN: at 0° the beam crosses its 2 mm thickness. NWA 7034 is a **breccia**:
fragments of rocks and minerals cemented in a fine-grained matrix. What is in it:

| Material | What it is | Size |
|---|---|---|
| matrix | fine grains (pyroxene, plagioclase, Fe oxides, phosphate), mixed below the resolution | — |
| plagioclase, pyroxene | mineral clasts (light and green on the drawing) | 0.1 to 0.5 mm, about 5 % of the volume each |
| basaltic clast | a fragment of lava: pyroxene with laths of plagioclase | about 1 mm |
| Fe-Ti oxides | grains of magnetite and ilmenite | 3 % of the volume, mostly very fine |
| apatite, pyrite | a few small grains | below 0.1 mm |
| pores, a crack | empty | up to 0.1 mm; the crack 1 mm long |
| H-rich Fe-oxyhydroxides | inside small clasts of 0.1 to 0.4 mm | 0.4 % of the volume, in clusters: **to be found** |

The amounts, sizes and attenuations of the clasts that hold hydrogen are those measured on this piece by Naver *et
al.*; the other phases follow what is published on NWA 7034.

The camera is just behind the sample and looks along the beam: horizontal axis $x$, vertical axis $y$, field 14 mm.

In [ ]:
spec = odin.sample_spec(SAMPLE)
smp_out = w.Output()
import sample_library as sl
with into(smp_out):
    sl.draw("odin", spec, hidden=["inclusions"], fig=plt.figure(figsize=(10, 3.6)))
    sl.show()
display(smp_out)

## 1. Operate the instrument: radiographs

ODIN is set for **white-beam imaging**: the choppers are stopped (all the wavelengths, the highest flux), and the images
are taken with a camera that integrates the light of a scintillator screen. Two setups:

| Setup | Camera | Pinhole D | Blur | Neutrons per pixel and second (at 5 MW) |
|---|---|---|---|---|
| **standard** | 128 × 128 pixels of 110 µm, thick bright scintillator | 3 cm | about 0.17 mm | about 2200 |
| **high resolution** | 512 × 512 pixels of 27.5 µm, thin scintillator | 1 cm | about 0.05 mm | about 24 |

At the design power of ESS, 5 MW (the line *Source power* above the controls says which power you have); at 2 MW,
2.5 times fewer.

**White beam.** No chopper selects the neutrons: all the wavelengths arrive together, as in a polychromatic X-ray
beam, and the camera adds them up. That gives the most neutrons, so the shortest exposures; the price is that the
images do not know the wavelength ($\Sigma$ is an average over the spectrum). Sorting the neutrons by wavelength (the
time of flight, used in other exercises) needs the choppers: they keep the slow neutrons of one pulse from mixing with
the fast ones of the next, and let far fewer neutrons through.

The blur comes mostly from the pinhole (the scintillator adds its own, 0.1 mm or 0.02 mm): seen from the camera, 5 cm behind the sample, the pinhole 10.5 m away makes
each point of the sample a spot of size $l \cdot D / L$ (part 3 of the introduction to ODIN). The clasts you are
looking for are 0.1 to 0.4 mm.

**Why three images?** The camera does not measure the transmission $T$ directly: it counts, in each pixel, what
arrives. To get $T$, three images are needed:

- the **sample** image $S$: the neutrons transmitted through the sample;
- the **open beam** $O$ (sample out: stage at 500 mm): the same beam without the sample. The beam is not perfectly even over the
  field of view, and the pixels are not all equally sensitive: the open beam records both, so that they can be divided out;
- the **dark** $D$ (shutter closed, no neutrons): the camera gives a small signal even without neutrons (its offset and
  noise), which must be taken away from the two others.

Then, for each pixel, with each image counted per second of exposure,

$$T = \frac{S - D}{O - D}$$

Everything that is not the sample cancels: $T$ = 1 where nothing is stopped, $T$ = 0 where everything is stopped.

**The measurements to take**, in this order (set the controls below, then *Take image*):

| Step | Image | Setup | Shutter | Sample | Exposure (s) |
|---|---|---|---|---|---|
| A1 | dark | standard | closed | in | 10 |
| A2 | open beam | standard | open | out | 10 |
| A3 | sample at 0° | standard | open | in | 10 |
| B1 | dark | high resolution | closed | in | 300 |
| B2 | open beam | high resolution | open | out | 1500 |
| B3 | sample at 0° | high resolution | open | in | 300 |

**A (A1 to A3) is a first look**, quick, with the standard setup. **B (B1 to B3) is the real measurement**, with
the high-resolution setup. The open beam B2 is long on purpose: it divides every image, so its own noise would be
printed on all of them (part 4). The title of each image is filled in for you if you leave it empty.

In [ ]:
st = odin.status()
setup = w.ToggleButtons(options=list(SETUPS), value="standard", description="Setup")
shut = w.ToggleButtons(options=[("closed", "closed"), ("open", "open")], value=st["shutter"], description="Shutter")
inout = w.ToggleButtons(options=[("in", 0.0), ("out", 0.5)], value=0.0 if abs(st["stage_x"]) < 0.02 else 0.5,
                        description="Sample")
expo = w.Dropdown(options=[10, 30, 300, 1500], value=10, description="Exposure (s)")
title = w.Text(value="", placeholder="e.g. open beam", description="Title")
button = w.Button(description="Take image", button_style="primary", icon="camera", layout=w.Layout(width="auto"))
out = w.Output()
img_take = live_view.FigureImage()

def default_title():
    if shut.value == "closed":
        return f"dark ({setup.value})"
    return f"open beam ({setup.value})" if inout.value > 0.02 else f"meteorite 0° ({setup.value})"

def take(_):
    # runs in a thread (measure_control): the settings are locked meanwhile, Stop cancels; prints go to ctl.status
    try:
        px, D = SETUPS[setup.value]
        odin.move(choppers="white", pinhole=D, diffuser="in", shutter=shut.value, stage_x=inout.value,
                  stage_rot=0.0, sample=SAMPLE)
        print("Taking the image (the first one takes a few seconds: the beam is prepared)...")
        f = odin.camera(float(expo.value), title=title.value or default_title(), pixels=px)
        ctl.clear()
        print(f"Written: {os.path.basename(f)} ({title.value or default_title()}).")
        with img_take:
            fig, ax = plt.subplots(figsize=(5.2, 4.4))
            im = ax.imshow(odin.last_image, origin="lower", extent=EXT, cmap="gray")
            fig.colorbar(im, ax=ax, label="counts per pixel")
            ax.set(xlabel="x (mm)", ylabel="y (mm)", title=f"{os.path.basename(f)}: {title.value or default_title()}")
            plt.tight_layout()
    except Cancelled:
        pass                                  # Control says it: nothing was recorded
    except Exception as e:
        print("Error:", e)
    refresh_list()
    rename_update()

import source_power                          # the power of the source: read-only (the teacher, or the app)
power = source_power.control(odin)
ctl = mc.Control(odin, button, lock=[setup, shut, inout, expo, title] + source_power.lockable(power))
ctl.on_click(take)
display(w.VBox([setup, shut, inout, expo, power, title, w.HBox([button, ctl.stop]), ctl.status, img_take]))

## 2. Your measurements

Each image is a numbered file, as on the real instrument; the table is your logbook.

In [ ]:
listing = w.Output()

def measurements():
    ms = []
    if os.path.isfile(odin.log_file):
        for l in open(odin.log_file, encoding="utf-8"):
            e = json.loads(l)
            if e["action"] != "camera" or not e["details"].get("file"):
                continue
            d = e["details"]; s = d["state"]
            if s.get("sample") != SAMPLE:
                continue
            kind = "dark" if s["shutter"] == "closed" else ("open beam" if abs(s["stage_x"]) > 0.02 else "sample")
            px = d.get("pixels", 128)
            ms.append(dict(file=d["file"], path=os.path.join(odin.folder, d["file"]), title=d["title"], t=d["t"],
                           kind=kind, rot=float(s["stage_rot"]), counts=d["n_counts"], pixels=px,
                           setup="high resolution" if px == 512 and abs(s["pinhole"] - 0.01) < 1e-6 else
                                 ("standard" if px == 128 and abs(s["pinhole"] - 0.03) < 1e-6 else "other")))
    return ms

def label(m):
    return f"{m['file']}: {m['title'] or '(no title)'} ({m['t']:g} s)"

def refresh_list():
    ms = measurements()
    of = lambda kind: [(label(m), m["path"]) for m in reversed(ms) if m["kind"] == kind and m["setup"] == show_setup.value
                       and not m["title"].startswith("tomography")]
    for dd, opts in ((pick_S, of("sample")), (pick_O, of("open beam")), (pick_D, of("dark"))):
        dd.options = opts
        if opts:
            dd.value = opts[0][1]
    with into(listing):
        n_tomo = sum(m["title"].startswith("tomography") for m in ms)
        if not ms:
            print("No image yet.")
        for m in ms:
            if not m["title"].startswith("tomography"):
                print(f"{m['file']}  {m['title'] or '(no title)':34s} {m['kind']:10s} t = {m['t']:5g} s  "
                      f"{m['setup']:16s} -> {m['counts']:,} counts")
        if n_tomo:
            print(f"... and {n_tomo} images of tomography (part 4)")

wide = w.Layout(width="560px")
show_setup = w.ToggleButtons(options=list(SETUPS), value="standard", description="Setup")
pick_S = w.Dropdown(description="Radiograph", layout=wide)
pick_O = w.Dropdown(description="Open beam", layout=wide)
pick_D = w.Dropdown(description="Dark", layout=wide)
show_setup.observe(lambda c: refresh_list(), "value")
refresh_list()
rename, rename_update = mc.rename_box(odin, refresh=refresh_list)
display(w.VBox([listing, rename]))

## 3. The radiograph

*Show the transmission* computes $T = (S - D)/(O - D)$ for each pixel (each image per second of exposure). Choose the
setup of the images to compare. The profile follows one horizontal line of pixels (choose its height).

In [ ]:
height = w.FloatSlider(value=2.86, min=-6.5, max=6.5, step=0.05, description="Line y (mm)", continuous_update=True,
                       layout=w.Layout(width="420px"))
button_T = w.Button(description="Show the transmission", button_style="success", icon="image", layout=w.Layout(width="auto"))
out_T = w.Output()
img_T = live_view.FigureImage()

def chosen():
    if not (pick_S.value and pick_O.value):
        raise ValueError("Take first an open beam and a radiograph of the meteorite with this setup (part 1).")
    return pick_S.value, pick_O.value, pick_D.value

def show_T(_=None):
    with out_T:
        clear_output()
        try:
            S, O, D = chosen()
            T = tomo.transmission(S, O, D)
            n = T.shape[0]
            with img_T:
                fig, (a, b) = plt.subplots(1, 2, figsize=(11, 4.4), gridspec_kw=dict(width_ratios=[1, 1.25]))
                im = a.imshow(T, origin="lower", extent=EXT, cmap="gray", vmin=0.8, vmax=1.0)
                fig.colorbar(im, ax=a, label="transmission T")
                a.axhline(height.value, color="tab:orange", lw=1)
                a.set(xlabel="x (mm)", ylabel="y (mm)", title=f"Transmission ({show_setup.value}, {n} × {n} pixels)")
                row = int(np.clip(round(height.value / tomo.pixel_mm(n) + (n - 1) / 2), 0, n - 1))
                x = np.linspace(EXT[0], EXT[1], n)
                b.plot(x, T[row], color="tab:orange", lw=0.8)
                b.set(xlabel="x (mm)", ylabel="transmission T", title=f"Profile at y = {height.value:.2f} mm",
                      ylim=(0.75, 1.05), xlim=(-5, 5))
                b.grid(alpha=0.3)
                plt.tight_layout()
            if D is None:
                print("Note: no dark image, the dark signal of the camera is not subtracted.")
        except Exception as err:
            clear_output()
            print("Not possible yet:", err)

button_T.on_click(show_T)
height.observe(live_view.throttled(lambda: show_T() if img_T.value else None), "value")
display(w.VBox([show_setup, pick_S, pick_O, pick_D, w.HBox([button_T, height]), out_T, img_T]))

In [ ]:
q.choice("With the **standard** setup, can the radiograph show the clasts of 0.1 to 0.4 mm?",
         ["Hardly: they are as small as the blur (0.17 mm) and the pixels (0.11 mm), and their contrast of a few % is "
          "lost in the noise.",
          "Yes, clearly: neutrons see hydrogen very well.",
          "No: neutrons do not see rocks.",
          "Yes, but only at 90°."],
         answer=0, why={1: "Hydrogen gives contrast, but the clasts are tiny and the 2 mm of rock around them is added "
                           "on each line of sight.",
                        2: "The rock lets 90 % of the neutrons through: it is seen, as a uniform grey.",
                        3: "At 90° the beam crosses 8 mm of rock: even less contrast for a small clast."},
         explanation="To see a 0.2 mm clast, the blur and the pixels must be well below 0.2 mm, and enough neutrons must "
                     "be counted to see a contrast of a few %: hence the high-resolution setup and long exposures.",
         hints=["Compare the size of the clasts with the blur and the pixels of the table of part 1."],
         title="Q1. The standard setup")

In [ ]:
q.number("The blur of the high-resolution setup: $l \\cdot D / L$ with $l$ = 5 cm (sample to camera), D = 1 cm, "
         "L = 10.5 m. How many micrometers?", value=48, tol=3, unit="µm",
         mistakes=[(143, "this is the blur with the 3 cm pinhole"), (0.048, "right, but in mm; the answer is asked in µm")],
         hints=["5 cm × 1 cm / 1050 cm, in cm; then × 10 000 for µm."],
         explanation="0.05 m × 0.01 m / 10.5 m = 4.8·10⁻⁵ m = 48 µm. Three times sharper than with 3 cm, but 6 times "
                     "fewer neutrons (9 by the area of the pinhole; but the guide focuses the beam onto it, brightest at the center, so the edge of the 3 cm hole is weakly lit): longer exposures.", title="Q2. The blur")

Curious about what your files really contain, and how the transmission is computed from them? Open the section below.

In [ ]:
analysis_chain.section("odin_camera", odin)

**Reading the radiograph.** About 90 % of the neutrons cross the 2 mm of rock: rocks are almost transparent to
neutrons. With the standard setup the piece looks uniform: the clasts are as small as the blur and the pixels, and
their contrast (a few %) is hidden in the noise. With the high-resolution setup small darker and brighter specks
appear, but a radiograph adds up the whole 2 mm: a speck can be one grain or several, near the front or the back.

## 4. The tomography

*Run the tomography* turns the sample in steps of 3° over 180° and takes one high-resolution image of 300 s at each
angle: 60 images, **5 hours of beam** (the twin computes them in a few minutes). Then, for each horizontal row of
pixels, the projections $-\ln T$ at all the angles are **filtered** and **smeared back** along the direction they were
taken in (filtered back-projection): this gives $\Sigma$ at each point of the slice. For comparison, Naver *et al.*
took 1125 images of 4 minutes at ICON (75 hours), with pixels of 6.2 µm.

**While it runs** (a few minutes here), read *Look inside* below: it tells you what to do with the slices.

In [ ]:
button_tomo = w.Button(description="Run the tomography", button_style="primary", icon="sync", layout=w.Layout(width="auto"))
bar = viz.Progress(60, "Images")            # images done, time elapsed, time left
out_tomo = w.Output()
REC = {}

def run_tomo(_):
    # runs in a thread (measure_control): Stop ends the series after the image under way (the images taken stay)
    try:
        show_setup.value = "high resolution"
        refresh_list()
        _, O, D = chosen()
        angles = np.arange(0, 180, 3.0)
        bar.start(len(angles))
        px, Dp = SETUPS["high resolution"]
        odin.move(choppers="white", pinhole=Dp, diffuser="in", shutter="open", stage_x=0.0, sample=SAMPLE)
        files = odin.tomography(angles, 300.0, title="tomography", pixels=px,
                                progress=bar.update)
        print("Filtering the projections...")
        REC["rec"] = tomo.Reconstruction(files, O, D)
        ctl_tomo.clear()
        print(f"{len(angles)} images reconstructed. Explore the slices below.")
        show_slices()
    except Cancelled:
        print("Tomography stopped: the images already taken stay in your files, but there is no reconstruction.")
    except Exception as err:
        print("Not possible yet:", err)
    finally:
        odin.move(stage_rot=0.0)
    refresh_list()
    rename_update()

ctl_tomo = mc.Control(odin, button_tomo, stopped=None)      # run_tomo says what was kept
ctl_tomo.on_click(run_tomo)
display(w.VBox([w.HBox([button_tomo, ctl_tomo.stop]), bar.box, ctl_tomo.status]))

**Look inside.** Left: a **horizontal slice** at the height $y$ you choose, seen from above ($x$ across, $z$ the depth
along the beam at 0°). Right: a **vertical slice** at the depth $z$ you choose (as a radiograph, but only that slice).
The color is $\Sigma$ in cm⁻¹. The rings centered on the rotation axis are an artifact: the noise of the open beam,
the same in every projection. Move the sliders through the piece; the largest clasts are near the top.

In [ ]:
ys = w.FloatSlider(value=2.86, min=-6.0, max=6.0, step=0.0275, description="Height y (mm)", continuous_update=True,
                   readout_format=".2f", layout=w.Layout(width="460px"))
zs = w.FloatSlider(value=0.0, min=-1.0, max=1.0, step=0.0275, description="Depth z (mm)", continuous_update=True,
                   readout_format=".2f", layout=w.Layout(width="460px"))
xs = w.FloatSlider(value=0.0, min=-4.0, max=4.0, step=0.0275, description="Point x (mm)", continuous_update=True,
                   readout_format=".2f", layout=w.Layout(width="460px"))
img_sl = live_view.FigureImage()
txt_sl = w.HTML("Run the tomography first.")

def show_slices(_=None):
    if "rec" not in REC:
        return
    r = REC["rec"]
    h, v = r.horizontal(ys.value), r.vertical(zs.value)
    with img_sl:
        fig, (a, b) = plt.subplots(1, 2, figsize=(12, 5.2), gridspec_kw=dict(width_ratios=[1.6, 1]))
        kw = dict(origin="lower", extent=EXT, cmap="magma", vmin=0, vmax=2.0)
        im = a.imshow(h, **kw)
        a.set(xlim=(-4.6, 4.6), ylim=(-1.6, 1.6), xlabel="x (mm)", ylabel="depth z (mm)",
              title=f"Horizontal slice at y = {ys.value:.2f} mm")
        a.axhline(zs.value, color="w", lw=0.6, ls="--"); a.plot(xs.value, zs.value, "c+", ms=14, mew=1.5)
        b.imshow(v, **kw)
        b.set(xlim=(-4.6, 4.6), ylim=(-6.6, 6.6), xlabel="x (mm)", ylabel="y (mm)",
              title=f"Vertical slice at depth z = {zs.value:.2f} mm")
        b.axhline(ys.value, color="w", lw=0.6, ls="--"); b.plot(xs.value, ys.value, "c+", ms=10, mew=1.5)
        fig.colorbar(im, ax=[a, b], label="Σ (cm⁻¹)", shrink=0.8)
    txt_sl.value = "Move the cross (+) onto a bright spot with the three sliders: part 5 measures Σ there."

for s_ in (ys, zs, xs):
    s_.observe(live_view.throttled(show_slices), "value")
display(w.VBox([ys, zs, xs, img_sl, txt_sl]))
show_slices()

In [ ]:
q.choice("Why must the **open beam** be measured much longer than each image of the tomography?",
         ["Every image is divided by the same open beam: its noise is printed identically on all of them and becomes "
          "rings around the rotation axis.",
          "Because the open beam is brighter.",
          "To use more beam time.",
          "It does not matter."],
         answer=0, why={1: "It is brighter, which helps, but the point is that it is the same in every projection.",
                        2: "Beam time is precious: here it is well spent.",
                        3: "Look at the rings in the slices of part 4."},
         explanation="A noise that changes from image to image averages out in the reconstruction; a noise that is the "
                     "same in all the images (the open beam's, a dead pixel) does not: it draws circles. Long open beams "
                     "(or several) are the standard remedy.", hints=["What do all the projections have in common?"],
         title="Q3. The rings")

In [ ]:
q.number("Σ of the **matrix** in the slices (cm⁻¹, in ODIN's white beam):", value=0.53, tol=0.08, unit="cm⁻¹",
         hints=["Part 5 prints it around the cross; or read the color of the slices far from the bright spots."],
         explanation="About 0.51–0.55 cm⁻¹ in ODIN's beam (Naver et al.: 0.45 cm⁻¹, taken by the twin at "
                     "1.8 Å; slower neutrons are attenuated more).",
         title="Q4. The matrix")

In [ ]:
q.choice("Where is the hydrogen?",
         ["In small bright regions (0.1–0.4 mm) grouped in clusters, not spread evenly.",
          "Evenly in the whole piece.",
          "Only at the surface.",
          "In the large dark regions."],
         answer=0, why={1: "The matrix is uniform, but look at the bright spots.",
                        2: "The bright spots are inside, at all depths.",
                        3: "Dark means a small Σ: little attenuation (feldspar)."},
         explanation="Naver et al. found the H-rich phases in localised macroscopic clusters, 0.4 % of the volume: a "
                     "new kind of water reservoir in the Martian crust.", hints=["Move the height slider through the piece."],
         title="Q5. Where?")

## 5. What is in the bright spots?

*Measure Σ* gives the mean $\Sigma$ in a small ball (0.06 mm radius) around the cross (+) of part 4, and in the matrix
around it (0.4 to 0.6 mm away, the median). Their **ratio** compares the spot with the matrix, almost independently of
the spectrum of the beam. Naver *et al.* computed $\Sigma$ of the minerals of NWA 7034 from their composition (their
Table S1, at 25 meV) and measured the matrix at ICON (0.45 cm⁻¹):

| Material (Naver *et al.*, Table S1) | Σ at 25 meV (cm⁻¹) | ratio to the matrix |
|---|---|---|
| plagioclase (albite, anorthite) | 0.27 – 0.28 | 0.6 |
| pyroxenes (enstatite, diopside, Ca-Fe-Al clinopyroxene) | 0.34 – 0.38 | 0.8 |
| chlorapatite | 0.34 | 0.8 |
| ilmenite | 0.70 | 1.6 |
| magnetite, maghemite | 0.80 – 0.82 | 1.8 |

In [ ]:
button_s = w.Button(description="Measure Σ", button_style="success", icon="crosshairs", layout=w.Layout(width="auto"))
out_s = w.Output()
RES = {}

def measure(_=None):
    with out_s:
        clear_output()
        if "rec" not in REC:
            print("Run the tomography first (part 4)."); return
        r = REC["rec"]
        spot = r.value(xs.value, ys.value, zs.value, 0.06)
        ang = np.linspace(0, 2 * np.pi, 12, endpoint=False)
        ring = [r.value(xs.value + d * np.cos(a), ys.value, float(np.clip(zs.value + d * np.sin(a), -0.85, 0.85)), 0.04)
                for d in (0.4, 0.6) for a in ang]
        mat = float(np.median(ring))
        RES.update(spot=spot, matrix=mat)
        print(f"at x = {xs.value:.2f}, y = {ys.value:.2f}, z = {zs.value:.2f} mm:  Σ = {spot:.2f} cm⁻¹")
        print(f"matrix around it:                       Σ = {mat:.2f} cm⁻¹")
        print(f"ratio spot / matrix = {spot / mat:.1f}")

button_s.on_click(measure)
display(w.VBox([button_s, out_s]))

In [ ]:
q.number("The ratio Σ(spot) / Σ(matrix) of the brightest spot you found:", value=3.5, tol=1.5,
         hints=["Put the cross on the brightest spot of the largest clast (near y = 2.86 mm), then Measure Σ."],
         explanation="About 3.5 in the middle of the largest H-rich region; its true value is 1.9 / 0.45 = 4.2 (Naver "
                     "et al., ICON), but the region is only 0.12 mm thin in one direction and the blur mixes it with "
                     "its surroundings; a smaller spot gives less.", title="Q6. The ratio")

In [ ]:
q.choice("Which material can attenuate **3 to 4 times** more than the matrix?",
         ["Only a hydrogen-bearing one: no anhydrous mineral of the table reaches twice the matrix.",
          "Magnetite: it is the densest.",
          "Feldspar.",
          "Any mineral, if the grain is large enough."],
         answer=0, why={1: "Magnetite is at 1.8 times the matrix, not 3 or 4.",
                        2: "Feldspar attenuates less than the matrix (0.6 times).",
                        3: "Σ does not depend on the size of the grain: it is a property of the material."},
         explanation="This is the argument of Naver et al.: the measured attenuation of the H-rich regions (1.9 cm⁻¹) "
                     "exceeds the theoretical value of their minerals without hydrogen (0.61 cm⁻¹); only hydrogen can "
                     "explain the difference.", hints=["Compare your ratio with the last column of the table of part 5."],
         title="Q7. Which element?")

**Reading the numbers.** The values of the twin are higher than those of the table: the table is for neutrons
of 1.8 Å (25 meV), the neutrons counted on ODIN have a mean wavelength of 3.3 Å, and these materials attenuate slower
neutrons more. Ratios change much
less. The tomography also blurs a spot of 0.1 to 0.3 mm with its surroundings (the blur, about 0.05 mm, and the
pixels, 0.0275 mm): the value in the middle of a small spot is lower than the true one.

## 6. How much water?

Naver *et al.* "assume that all excess measured neutron attenuation results from the presence of hydrogen". The
parts of the clasts without hydrogen show $\Sigma_\text{clast}$ = 0.57 cm⁻¹ at ICON, that is **0.69 cm⁻¹** in ODIN's
beam (the same minerals, computed with the spectrum of the twin), and liquid water **4.9 cm⁻¹**. The fraction of the
volume of the spot that water would fill is then

$$f = \frac{\Sigma_\text{spot} - \Sigma_\text{clast}}{\Sigma_\text{water}}$$

and, with the density of the region (about 3.0 g/cm³), the water by mass is $f \times 1.0 / 3.0$.

In [ ]:
button_w = w.Button(description="Water in the spot", button_style="success", icon="tint", layout=w.Layout(width="auto"))
out_w = w.Output()
SIG_CLAST, SIG_WATER, RHO = 0.69, 4.9, 3.0     # effective_sigma in the camera spectrum (white, 1 cm pinhole)

def water(_=None):
    with out_w:
        clear_output()
        if "spot" not in RES:
            print("Measure Σ in a bright spot first (part 5)."); return
        f = (RES["spot"] - SIG_CLAST) / SIG_WATER
        print(f"f = ({RES['spot']:.2f} − {SIG_CLAST}) / {SIG_WATER} = {f:.2f}  ->  water by mass = {100 * f / RHO:.0f} %")
        if f <= 0:
            print("No excess attenuation here: is the cross on a bright spot?")

button_w.on_click(water)
display(w.VBox([button_w, out_w]))

In [ ]:
q.number("The water by mass in the brightest spot (part 6, %):", value=8, tol=6, unit="%",
         hints=["Part 6 computes it from your Σ of part 5."],
         explanation="The model holds about 14 % (Naver et al.: up to 15 wt% OH). The blur lowers Σ in the middle of a "
                     "small region, so the estimate is lower: a few % to about 12 %.", title="Q8. How much water?")

In [ ]:
q.number("Naver et al. estimate that these clasts hold 635 ppm of water, of the 6000 ppm of the whole meteorite. What "
         "share is that (%)?", value=10.6, tol=1.0, unit="%",
         hints=["635 / 6000."], explanation="635 / 6000 = 10.6 %, the 11 % of the paper: a macroscopic reservoir, while "
                                            "most of the water is spread in the fine matrix.", title="Q9. Their share")

In [ ]:
q.choice("**Your report to the Mars team.**",
         ["The water is concentrated in small clasts (0.1–0.4 mm) grouped in clusters; their attenuation can only be "
          "explained by hydrogen, about 10 % water by mass; the tomography gives their positions for sampling.",
          "The water is spread evenly in the meteorite.",
          "There is no water: the bright spots are magnetite.",
          "Neutrons cannot answer: X-rays are needed."],
         answer=0, why={1: "The bright spots are local.", 2: "Magnetite reaches 1.8 times the matrix, not 4.",
                        3: "Neutrons are the right probe for hydrogen; X-rays see the Fe-Ti oxides."},
         explanation="Your conclusion is that of Naver et al. (2026): a new macroscopic crustal water reservoir on early "
                     "Mars, found without cutting the meteorite.", hints=["Q5, Q7 and Q8 together."], title="Q10. The report")

**Limits of the twin.** The piece is a model: its size, the amounts, sizes and attenuations of its H-rich clasts
(each of the three largest with its own measured Σ) are those published by Naver *et al.*; their tomograms are not
public yet, so where the clusters and clasts are is drawn at random (fixed), as are the lumpy shapes of the grains.
The other phases (mineral and basaltic clasts, pores, a crack) follow the literature on NWA 7034. The hydrogen is
represented by an equivalent amount of water (NCrystal). The minerals are mixed at the scale of
each region (no crystal, no Bragg edge). The camera is modeled by its mean signal, its noise and its blur.

## 7. Bonus round: for X-ray experts

For those who know X-ray imaging or diffraction: click on the title below to open it.

In [ ]:
from page_output import collapsed
from IPython.display import Markdown
with collapsed("Bonus round: for X-ray experts (click to open)"):
    display(Markdown("You know X-ray imaging or diffraction? Then this round is for you: every question puts what you "
                     "just did with neutrons next to what you would do with X-rays, with numbers. They count in your "
                     "score like the others. Values used: neutrons from NCrystal, as in the twin; X-rays from the NIST "
                     "tables (XCOM)."))
    q.number("1 mm of water in the beam. With neutrons of 3 Å (the middle of ODIN's beam) its attenuation coefficient is "
             "Σ = 4.9 cm⁻¹ (hydrogen); with X-rays of 60 keV, μ = 0.21 cm⁻¹. Which fraction of the **neutrons** does it "
             "remove, 1 − T (%)?",
             value=39.0, tol=1.5, unit="%",
             mistakes=[(2.0, "that is the X-ray value (μ = 0.21 cm⁻¹); the question is about neutrons"),
                       (49, "4.9 cm⁻¹ × 0.1 cm = 0.49 is −ln T, not 1 − T: T = e^(−0.49)")],
             hints=["T = exp(−Σ d), with d = 0.1 cm.", "1 − exp(−0.49)."],
             explanation="1 − e^(−0.49) = 39 % with neutrons, 1 − e^(−0.021) = 2 % with X-rays. And 1 mm of iron: neutrons "
                         "10 % (Σ = 1.1 cm⁻¹ at 3 Å), X-rays of 60 keV 61 % (μ = 9.5 cm⁻¹). The contrasts are very "
                         "different: X-rays see the heavy elements, neutrons see the hydrogen.", title="X1. Water, for neutrons and X-rays")
    q.number("An H-rich clast of 0.3 mm in the matrix of the meteorite (ICON, 1.8 Å: Σ = 1.9 cm⁻¹ against 0.45 cm⁻¹). By how "
             "much does it lower the transmission of a line that crosses it, compared with the matrix around (%)?",
             value=4.3, tol=0.4, unit="%",
             mistakes=[(14.5, "(1.9 − 0.45) × 0.1: the clast is 0.3 mm = 0.03 cm thick, not 1 mm")],
             hints=["Only the difference of Σ matters, over the thickness of the clast.", "1 − exp(−(1.9 − 0.45) × 0.03)."],
             explanation="1 − e^(−1.45 × 0.03) = 4.3 %: a small contrast in one radiograph, where the clasts also overlap "
                         "the matrix in front and behind. The tomography separates them, and each voxel then holds the full "
                         "ratio of Σ (4 times the matrix). With X-rays the same clasts stand out by their iron, not by their "
                         "hydrogen: they look like the Fe-Ti oxides.", title="X2. The contrast of a clast")
    q.choice("A laboratory micro-CT magnifies the sample by bringing it close to a small focal spot (a cone beam). Why does "
             "ODIN put the camera **5 cm** behind the sample instead, without magnification?",
             ["The neutron source seen by the sample is the pinhole, 1 to 4 cm wide: a magnifying geometry would multiply its "
              "blur. ODIN keeps the camera close and the beam almost parallel (L/D up to 1000); the resolution is set by "
              "L/D and by the detector.",
              "Neutrons have no charge, so they cannot be magnified.",
              "There is no room behind the sample on ODIN.",
              "The camera pixels are small enough: magnification would not help."],
             answer=0, why={1: "X-ray micro-CT does not use lenses either: its magnification is geometric (cone beam), and "
                               "neutrons would follow the same geometry. The problem is the size of the source.",
                            2: "Room is not the reason: the blur is (part 3 of the introduction: blur = l · D / L).",
                            3: "Pixels of 27.5 µm are not the limit: with the camera 1 m away the 1 cm pinhole would blur by "
                               "1 mm."},
             explanation="Geometric blur = l · D / L. With a focal spot of a few µm, X-rays can afford a large l (and "
                         "magnify); with a pinhole of 1 cm at 10.5 m, every cm of l adds 10 µm of blur: the camera stays "
                         "close, and the parallel-beam reconstruction (180°) applies.", title="X3. No magnification")
    q.choice("For X-rays in a white beam, a thick sample **hardens** the beam (the soft photons are absorbed first), which "
             "biases the reconstruction. What happens to a white neutron beam crossing the hydrated meteorite?",
             ["The long wavelengths are removed more (absorption grows as λ, the scattering by hydrogen too): the mean "
              "wavelength goes down, the analog of beam hardening.",
              "Nothing: neutrons are not absorbed.",
              "The short wavelengths are removed more, as for X-rays.",
              "Each neutron changes its wavelength in the sample."],
             answer=0, why={1: "Hydrogen and many nuclei remove neutrons strongly: that is the contrast of this exercise.",
                            2: "For neutrons it is the other way: the cross-sections grow with the wavelength (water: Σ = "
                               "3.6 cm⁻¹ at 1.8 Å, 4.9 cm⁻¹ at 3 Å).",
                            3: "Most neutrons that reach the camera crossed without exchanging energy; the attenuation "
                               "removes neutrons, it does not shift the wavelength of the others."},
             explanation="The attenuation depends on λ: a white-beam Σ is an average over the spectrum that changes with "
                         "the thickness crossed. ODIN can avoid it by selecting the wavelength (time of flight): one Σ per "
                         "wavelength.", title="X4. Beam hardening, for neutrons")
    q.number("**Message from the Mars team:** \"Your clasts are faint. Can you make them stand out more, without a longer "
             "count?\" ODIN can keep only one slice of wavelengths (time of flight). Σ (cm⁻¹) from NCrystal: matrix 0.40 at 1 Å, "
             "0.45 at 1.8 Å, 0.62 at 5 Å; H-rich clast 1.37 at 1 Å, 1.90 at 1.8 Å, 3.16 at 5 Å. At **5 Å**, by how much does a "
             "0.3 mm clast lower the transmission compared with the matrix around (%)? (At 1.8 Å it was 4.3 %, X2.)",
             value=7.3, tol=0.4, unit="%",
             mistakes=[(4.3, "that is 1.8 Å (X2); take the values at 5 Å"),
                       (22.4, "the clast is 0.3 mm = 0.03 cm thick, not 1 mm"),
                       (2.9, "that is 1 Å: shorter wavelengths give less contrast, not more")],
             hints=["As in X2: 1 − exp(−ΔΣ × d), with ΔΣ = Σ(clast) − Σ(matrix) at 5 Å and d = 0.03 cm.",
                    "ΔΣ = 3.16 − 0.62 = 2.54 cm⁻¹."],
             explanation="1 − e^(−2.54 × 0.03) = 7.3 %: almost twice the 4.3 % at 1.8 Å, and 2.5 times the 2.9 % at 1 Å. The "
                         "scattering by hydrogen grows fast toward long wavelengths, the matrix changes little (0.40 → 0.62). "
                         "For you it is like turning down the kV of an X-ray tube to gain contrast, but here it is the light "
                         "element, hydrogen, that gains.", title="X5. Tune the color of your neutrons")
    q.choice("A colleague is thrilled: \"Then let's image at **8 Å** only: the clasts give 9.1 %!\" What is the catch?",
             ["The pulse holds far fewer neutrons at 8 Å than at a few Å (the cold moderator gives few slow ones), and a thin slice of wavelengths keeps only a "
              "small part of the beam: for the same noise the count would be many times longer, for a contrast barely higher "
              "than at 5 Å.",
              "Neutrons of 8 Å do not cross the meteorite.",
              "The camera cannot detect neutrons of 8 Å.",
              "At 8 Å the neutrons are reflected by the surface of the meteorite."],
             answer=0, why={1: "Σ(matrix, 8 Å) = 0.73 cm⁻¹: 86 % of them cross the 2 mm piece.",
                            2: "It detects them even better: ε = 1 − e^(−8/8) = 63 %, against 20 % at 1.8 Å (introduction, part 6).",
                            3: "Reflection only happens at grazing angles on very flat surfaces (a neutron mirror), not here."},
             hints=["How many neutrons of 8 Å does the cold moderator of ESS give, compared with neutrons of 2 to 5 Å?",
                    "Contrast is not everything: a radiograph also needs neutrons."],
             explanation="What counts is the contrast compared with the noise, and the noise falls as 1/√N. From 5 to 8 Å the "
                         "contrast gains 25 %, but the neutrons become rare: the gain is eaten by the noise. The best choice "
                         "balances both, as you balance kV and flux with X-rays.", title="X6. The catch")

In [ ]:
from page_output import collapsed
from IPython.display import Markdown
with collapsed("Going further: harder questions (click to open)"):
    display(Markdown("**Going further (harder, optional).** Three questions that go beyond the page: the noise of an image, the reduction of ESS, the number of angles of a tomography. They count in your score like the others."))
    q.number("**Noise.** In one image of the tomography, a pixel of the open beam collects about 24 neutrons per second (5 MW); "
             "the rock lets 90 % of them through. Each image lasts 300 s and the open beam 1500 s. Counting noise: "
             "N counted neutrons fluctuate by √N. What is the relative noise of T = S / O in one pixel (%)? (Neglect the dark.)",
             value=1.35, tol=0.03, unit="%",
             mistakes=[(1.24, "that is the noise of the sample image alone; the open beam is counted too, and its noise adds"),
                       (1.18, "the rock lets only 90 % through: the sample image has 0.9 × 24 × 300 neutrons"),
                       (0.53, "that is the open beam alone; the sample image is noisier")],
             hints=["The relative noise of N counts is 1/√N. For a ratio, the relative noises add in quadrature.",
                    "√(1/N_S + 1/N_O), with N_S = 0.9 × 24 × 300 and N_O = 24 × 1500. Give two decimals."],
             explanation="N_S = 6480, N_O = 36000: √(1/6480 + 1/36000) = 1.35 %. A clast of 0.3 mm lowers T by "
                         "about 6 % in ODIN's white beam: 4.4 times the noise of one pixel, visible but grainy. It covers about 100 pixels: the "
                         "tomography, which combines all the pixels and all the angles, lowers the noise about 10 times. "
                         "This is why a long open beam matters: its noise is in every image.",
             title="X7. The noise of one pixel")
    q.number("**Reduction.** By mistake, the open beam was taken at 2 MW and the radiograph of the rock at 5 MW, with the "
             "same exposure. The rock lets 90 % through. If you divide each image by its exposure time only, which "
             "transmission do you find?",
             value=2.25, tol=0.05, unit="",
             mistakes=[(0.36, "the other way round: the sample image got 2.5 times more neutrons, not fewer"),
                       (0.9, "that is the true value, the one the ESS reduction finds: it divides by the proton charge, not "
                             "by the time. Here you divide by the time only")],
             hints=["At 5 MW each pulse carries 2.5 times more protons than at 2 MW, so 2.5 times more neutrons.",
                    "0.9 × 5 / 2."],
             explanation="0.9 × 2.5 = 2.25: a transmission above 1, impossible. The number of neutrons follows the protons on "
                         "the target, not the clock: the power changes, and even at constant power the pulses vary by about "
                         "1 % and the beam sometimes stops. So the ESS reduction (and the twin) divides each image by the "
                         "proton charge it received, written in the file: then T = 0.90 whatever the power.",
             title="X8. Per second or per proton?")
    q.number("**Tomography.** A rule of thumb: a tomography needs about (π/2) × N projections over 180°, N the number of "
             "pixels across the object, to sample the edge of the slices as finely as their middle. The piece is 8 mm wide "
             "and the pixels 27.5 µm. How many projections?",
             value=457, tol=15, unit="",
             mistakes=[(291, "that is N, the pixels across the piece; multiply by π/2"),
                       (914, "π × N: the rule is (π/2) × N over 180°"),
                       (60, "that is what you took; the question asks for the rule")],
             hints=["N = 8 mm / 27.5 µm.", "N = 291; then × π/2."],
             explanation="N = 8 / 0.0275 = 291, × π/2 = 457 projections. You took 60, to keep 5 hours of beam; 457 would take "
                         "38 hours. With too few angles the gaps between them show far from the rotation axis: streaks, "
                         "and fine details smeared at the edges of the slices, while the middle stays good. It is the usual "
                         "trade between beam time and quality; Naver et al. took 1125 images at ICON.",
             title="X9. How many angles?")

## 8. Your results

Your score on the questions of the page, and a report to download.

In [ ]:
q.report()

## 9. Submit your work

When you are done, tell what you thought of the exercise and submit your work.

In [ ]:
import submit
_ = submit.Submit("odin/exercises/meteorite_imaging", "bsc", quiz=q)